# 00 · Catálogo y schemas · Arquitectura medallón fenómeno Niño

Crea (si no existen) los schemas de Unity Catalog por capa y muestra el **inventario de tablas por capa**, de modo que al revisar el catálogo `workspace` se vea claramente qué tablas pertenecen a cada capa:

* `workspace.fenomeno_nino_bronze` → capa 01 Bronze (crudo)
* `workspace.fenomeno_nino_silver` → capa 02 Silver (limpieza/imputación)
* `workspace.fenomeno_nino_gold` → capa 03 Gold (modelo dimensional para BI)

## 1. Creación de schemas por capa

Idempotente: `CREATE SCHEMA IF NOT EXISTS` permite re-ejecutar este notebook sin error.

In [0]:
CATALOGO = "workspace"
CAPAS = {
    "01 · Bronze": "fenomeno_nino_bronze",
    "02 · Silver": "fenomeno_nino_silver",
    "03 · Gold": "fenomeno_nino_gold",
}
for capa, schema in CAPAS.items():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{schema}")
    print(f"OK · {CATALOGO}.{schema} → capa {capa}")

## 2. Inventario de tablas por capa

Vista consolidada desde `information_schema`: cada tabla del proyecto aparece agrupada por su capa. Al abrir **Catalog > workspace** verás los tres schemas, uno por capa.

In [0]:
display(spark.sql(f"""
  SELECT
    CASE table_schema
      WHEN 'fenomeno_nino_bronze' THEN '01 · Bronze'
      WHEN 'fenomeno_nino_silver' THEN '02 · Silver'
      WHEN 'fenomeno_nino_gold'  THEN '03 · Gold'
    END AS capa,
    table_schema AS schema,
    table_name AS tabla,
    table_type AS tipo
  FROM {CATALOGO}.information_schema.tables
  WHERE table_schema LIKE 'fenomeno_nino%'
  ORDER BY schema, tabla
"""))